In [2]:
import json
import pandas as pd

with open("mental.json", "r", encoding="utf-8") as df:
    data = json.load(df)

tables = {}

for item in data:
    if item.get("type") == "table":
        table_name = item["name"]
        tables[table_name] = pd.DataFrame(item["data"])

print(tables.keys())

dict_keys(['highlight_phq', 'highlight_range', 'post', 'post_tag', 'report_post', 'tag', 'user'])


In [3]:
post = tables["post"]
post_tag = tables["post_tag"]
user = tables["user"]
tag = tables["tag"]
report_post = tables["report_post"]

In [4]:
df = (
    post
    .merge(post_tag, left_on="id", right_on="post_id")
    .merge(tag, left_on="tag_id", right_on="id")
)

df = df[["txtContent", "tagName"]]

df.head()

,txtContent,tagName
0,NaN,ไม่แน่ใจ
1,NaN,ไม่แน่ใจ
2,NaN,ไม่แน่ใจ
3,NaN,ไม่แน่ใจ
4,NaN,ไม่แน่ใจ


In [12]:
df = df.dropna(subset=["txtContent"])
df = df[df["txtContent"].str.strip() != ""]
df = df.drop_duplicates(subset=["txtContent"])

In [13]:
import re

def clean_text(text):

    text = str(text)

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"www\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#\w+", "", text)

    text = re.sub(r"\n", " ", text)

    text = re.sub(r"\s+", " ", text)

    text = re.sub(r"[^\u0E00-\u0E7Fa-zA-Z0-9 ]", "", text)

    return text.strip()

df["clean_text"] = df["txtContent"].apply(clean_text)

In [14]:
from pythainlp.tokenize import word_tokenize

df["tokens"] = df["clean_text"].apply(
    lambda x: word_tokenize(x, engine="newmm")
)

from pythainlp.corpus.common import thai_stopwords

stopwords = set(thai_stopwords())

df["tokens"] = df["tokens"].apply(
    lambda words: [w for w in words if w not in stopwords]
)

df["processed_text"] = df["tokens"].apply(
    lambda x: " ".join(x)
)

In [17]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

tensor_size = (10000, 10000)  
a = torch.randn(tensor_size, device=device)  
b = torch.randn(tensor_size, device=device)  

c = a + b  

print("Result shape (moved to CPU for printing):", c.cpu().shape)

print("Current GPU memory usage:")
print(f"Allocated: {torch.cuda.memory_allocated(device) / (1024 ** 2):.2f} MB")
print(f"Cached: {torch.cuda.memory_reserved(device) / (1024 ** 2):.2f} MB")

Using device: cpu
Result shape (moved to CPU for printing): torch.Size([10000, 10000])
Current GPU memory usage:
Allocated: 0.00 MB
Cached: 0.00 MB


In [18]:
import torch
import torch.nn as nn

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 4)  
        self.fc2 = nn.Linear(4, 1)  

    def forward(self, x):
        x = torch.relu(self.fc1(x))  
        x = self.fc2(x)               
        return x

In [24]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label"] = le.fit_transform(df["tagName"])

In [22]:
from sklearn.model_selection import train_test_split

X = df["txtContent"]
y = df["tagName"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df["processed_text"])

y = df["label"]

print(X)
print(y)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 80944 stored elements and shape (2100, 5000)>
  Coords	Values
  (0, 2836)	1.0
  (1, 4236)	0.13727913603371972
  (1, 2173)	0.056436618970552924
  (1, 1593)	0.09393441556302035
  (1, 398)	0.1072674975738475
  (1, 521)	0.14585599045684747
  (1, 4523)	0.08465804592882333
  (1, 4013)	0.058868776496278835
  (1, 1546)	0.1426706016395178
  (1, 3458)	0.20697338640733273
  (1, 577)	0.20525203927546734
  (1, 6)	0.1143233836586011
  (1, 4819)	0.07605950572425656
  (1, 4100)	0.15777751668950118
  (1, 2099)	0.15098696060010386
  (1, 2946)	0.08576678520314962
  (1, 1887)	0.23634566433309326
  (1, 2918)	0.07484921526904297
  (1, 592)	0.13421280943874275
  (1, 3582)	0.11670648780830964
  (1, 216)	0.057532720926802984
  (1, 1764)	0.07210210306255263
  (1, 2939)	0.0997058888334852
  (1, 4816)	0.14415752232881357
  (1, 4257)	0.06003231544448029
  :	:
  (2098, 3114)	0.10862244814541408
  (2098, 4181)	0.14895984766750278
  (2098, 4918)	0.30840069

In [ ]:
import requests
import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================
# CONFIG
# ==========================
LMSTUDIO_URL = "http://localhost:1234/v1/chat/completions"
MODEL_NAME = "Typhoon2.1 Gemma3 4B"

INPUT_FILE = "mental.json"
OUTPUT_FILE = "mental_labeled.json"

SAVE_EVERY = 100

# ==========================
# Test LM Studio
# ==========================
print("🔌 Testing LM Studio...")

try:
    test = requests.get("http://localhost:1234/v1/models", timeout=5)
    print("✅ Connected!")
except Exception as e:
    print("❌ Cannot connect to LM Studio")
    print(e)
    exit()

# ==========================
# Load Dataset
# ==========================
print("📂 Loading dataset...")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

posts = None

for item in data:
    if item.get("type") == "table" and item.get("name") == "post":
        posts = item["data"]
        break

if posts is None:
    raise Exception("Post table not found!")

print(f"Loaded {len(posts)} posts")


label_map = {
    "0": "ปกติ",
    "1": "ไม่ปกติ",
    "2": "ไม่แน่ใจ"
}

results = []

# ==========================
# Label Posts
# ==========================

for i, item in enumerate(posts):

    post_id = item.get("id", "")
    raw_text = item.get("txtContent")

    if raw_text is None:
        continue

    text = str(raw_text).strip()

    if text == "":
        continue

    print(f"[{i+1}/{len(posts)}]")

    prompt = f"""
ตอบเพียงตัวเลขเดียว

0 = ปกติ
1 = ไม่ปกติ
2 = ไม่แน่ใจ

ข้อความ:
{text}
"""

    try:

        response = requests.post(
            LMSTUDIO_URL,
            json={
                "model": MODEL_NAME,
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                "temperature": 0,
                "max_tokens": 200
            },
            timeout=300
        )

        raw = response.json()

        if "choices" not in raw:
            raise ValueError(raw)

        choice = raw["choices"][0]
        msg = choice["message"]
        answer = (msg.get("content") or "").strip()

        if not answer:
            # fall back to reasoning_content if content is empty
            answer = (msg.get("reasoning_content") or "").strip()

        if not answer:
            print("DEBUG - full raw response:", json.dumps(raw, ensure_ascii=False, indent=2))
            raise ValueError("Empty response from model")

        # find the first character that's actually a valid label
        digit = next((ch for ch in answer if ch in label_map), None)

        if digit is None:
            raise ValueError(f"No valid label digit found in response: {answer!r}")

        label = label_map.get(digit, "ไม่แน่ใจ")

        item["label"] = label

        results.append(item)

        print("→", label)

    except Exception as e:

        print("ERROR:", e)

        item["label"] = "ERROR"

        results.append(item)

    # Save every 100 posts
    if (i + 1) % SAVE_EVERY == 0:

        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                results,
                f,
                ensure_ascii=False,
                indent=2
            )

        print(f"💾 Saved {i+1} posts")

# ==========================
# Save Final
# ==========================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n🎉 Finished!")
print(f"Saved to {OUTPUT_FILE}")

🔌 Testing LM Studio...
✅ Connected!
📂 Loading dataset...
Loaded 2132 posts
[15/2132]
→ ไม่แน่ใจ
[16/2132]
→ ปกติ
[17/2132]
→ ปกติ
[18/2132]
→ ปกติ
[19/2132]
→ ปกติ
[20/2132]
→ ปกติ
[21/2132]
→ ไม่ปกติ
[22/2132]
→ ปกติ
[23/2132]
→ ไม่ปกติ
[24/2132]
→ ไม่ปกติ
[25/2132]
→ ปกติ
[26/2132]
→ ไม่ปกติ
[27/2132]
→ ไม่ปกติ
[28/2132]
→ ไม่ปกติ
[29/2132]
→ ไม่ปกติ
[30/2132]
→ ไม่ปกติ
[31/2132]
→ ปกติ
[32/2132]
→ ปกติ
[33/2132]
→ ปกติ
[34/2132]
→ ไม่ปกติ
[35/2132]
→ ไม่ปกติ
[36/2132]
→ ไม่ปกติ
[37/2132]
→ ปกติ
[38/2132]
→ ปกติ
[39/2132]
→ ไม่ปกติ
[40/2132]
→ ปกติ
[41/2132]
→ ปกติ
[42/2132]
→ ไม่ปกติ
[43/2132]
→ ไม่ปกติ
[44/2132]
→ ไม่ปกติ
[45/2132]
→ ปกติ
[46/2132]
→ ปกติ
[47/2132]
→ ไม่ปกติ
[48/2132]
→ ปกติ
[49/2132]
→ ปกติ
[50/2132]
→ ปกติ
[51/2132]
→ ไม่ปกติ
[52/2132]
→ ปกติ
[53/2132]
→ ปกติ
[54/2132]
→ ปกติ
[55/2132]
→ ปกติ
[56/2132]
→ ปกติ
[57/2132]
→ ไม่ปกติ
[58/2132]
→ ปกติ
[59/2132]
→ ไม่ปกติ
[60/2132]
→ ไม่ปกติ
[61/2132]
→ ไม่ปกติ
[62/2132]
→ ปกติ
[63/2132]
→ ไม่ปกติ
[64/2132]
→ ไม่ปกติ
[6

KeyboardInterrupt: 

In [ ]:
import requests
import json

# ==========================
# CONFIG
# ==========================
LMSTUDIO_URL = "http://localhost:1234/v1/chat/completions"
MODEL_NAME = "Typhoon2.1 Gemma3 4B"

INPUT_FILE = "mental.json"
OUTPUT_FILE = "mental_labeled_50test.json"

SAVE_EVERY = 100

# ==========================
# Test LM Studio
# ==========================
print("🔌 Testing LM Studio...")

try:
    test = requests.get("http://localhost:1234/v1/models", timeout=5)
    print("✅ Connected!")
except Exception as e:
    print("❌ Cannot connect to LM Studio")
    print(e)
    exit()

# ==========================
# Load Dataset
# ==========================
print("📂 Loading dataset...")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

posts = None

for item in data:
    if item.get("type") == "table" and item.get("name") == "post":
        posts = item["data"]
        break

if posts is None:
    raise Exception("Post table not found!")

print(f"Loaded {len(posts)} posts")

# Only process the first 50 posts for testing
posts = posts[:50]
print(f"Testing with first {len(posts)} posts")


label_map = {
    "0": "ปกติ",
    "1": "ไม่ปกติ",
    "2": "ไม่แน่ใจ"
}

results = []

# ==========================
# Label Posts
# ==========================

for i, item in enumerate(posts):

    post_id = item.get("id", "")
    raw_text = item.get("txtContent")

    if raw_text is None:
        continue

    text = str(raw_text).strip()

    if text == "":
        continue

    print(f"[{i+1}/{len(posts)}]")

    prompt = f"""
ตอบเพียงตัวเลขเดียว

0 = ปกติ
1 = ไม่ปกติ
2 = ไม่แน่ใจ

ข้อความ:
{text}
"""

    try:

        response = requests.post(
            LMSTUDIO_URL,
            json={
                "model": MODEL_NAME,
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                "temperature": 0,
                "max_tokens": 200
            },
            timeout=300
        )

        raw = response.json()

        if "choices" not in raw:
            raise ValueError(raw)

        choice = raw["choices"][0]
        msg = choice["message"]
        answer = (msg.get("content") or "").strip()

        if not answer:
            # fall back to reasoning_content if content is empty
            answer = (msg.get("reasoning_content") or "").strip()

        if not answer:
            print("DEBUG - full raw response:", json.dumps(raw, ensure_ascii=False, indent=2))
            raise ValueError("Empty response from model")

        # find the first character that's actually a valid label
        digit = next((ch for ch in answer if ch in label_map), None)

        if digit is None:
            raise ValueError(f"No valid label digit found in response: {answer!r}")

        label = label_map.get(digit, "ไม่แน่ใจ")

        item["label"] = label

        results.append(item)

        print("→", label)

    except Exception as e:

        print("ERROR:", e)

        item["label"] = "ERROR"

        results.append(item)

    # Save every 100 posts
    if (i + 1) % SAVE_EVERY == 0:

        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                results,
                f,
                ensure_ascii=False,
                indent=2
            )

        print(f"💾 Saved {i+1} posts")

# ==========================
# Save Final
# ==========================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n🎉 Finished!")
print(f"Saved to {OUTPUT_FILE}")

🔌 Testing LM Studio...
✅ Connected!
📂 Loading dataset...
Loaded 2132 posts
Testing with first 50 posts
[15/50]
→ ไม่แน่ใจ
[16/50]
→ ปกติ
[17/50]
→ ปกติ
[18/50]
→ ปกติ
[19/50]
→ ปกติ
[20/50]
→ ปกติ
[21/50]
→ ไม่ปกติ
[22/50]
→ ปกติ
[23/50]
→ ไม่ปกติ
[24/50]
→ ไม่ปกติ
[25/50]
→ ปกติ
[26/50]
→ ไม่ปกติ
[27/50]
→ ไม่ปกติ
[28/50]
→ ไม่ปกติ
[29/50]
→ ไม่ปกติ
[30/50]
→ ไม่ปกติ
[31/50]
→ ปกติ
[32/50]
→ ปกติ
[33/50]
→ ปกติ
[34/50]
→ ไม่ปกติ
[35/50]
→ ไม่ปกติ
[36/50]
→ ไม่ปกติ
[37/50]
→ ปกติ
[38/50]
→ ปกติ
[39/50]
→ ไม่ปกติ
[40/50]
→ ปกติ
[41/50]
→ ปกติ
[42/50]
→ ไม่ปกติ
[43/50]
→ ไม่ปกติ
[44/50]
→ ไม่ปกติ
[45/50]
→ ปกติ
[46/50]
→ ปกติ
[47/50]
→ ไม่ปกติ
[48/50]
→ ปกติ
[49/50]
→ ปกติ
[50/50]
→ ปกติ

🎉 Finished!
Saved to mental_labeled_50test.json


In [1]:
import requests
import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================
# CONFIG
# ==========================
LMSTUDIO_URL = "http://localhost:1234/v1/chat/completions"
MODEL_NAME = "Typhoon2.1 Gemma3 4B"

INPUT_FILE = "mental.json"
OUTPUT_FILE = "mental_labeled.json"

SAVE_EVERY = 100

# How many requests to have in flight at once.
# LM Studio / llama.cpp typically processes one request at a time per GPU slot
# unless you've enabled parallel slots in server settings. If your server only
# has 1 slot, raising this won't speed up inference itself, but it still helps
# by removing network/HTTP round-trip idle time between calls. Try 4-8 and
# watch your GPU/CPU usage; increase if utilization stays low, decrease if you
# see errors or the server falls over.
MAX_WORKERS = 4

label_map = {
    "0": "ปกติ",
    "1": "ไม่ปกติ",
    "2": "ไม่แน่ใจ"
}

# Reuse one TCP connection pool instead of opening a new connection per request
session = requests.Session()
save_lock = threading.Lock()


# ==========================
# Test LM Studio
# ==========================
print("🔌 Testing LM Studio...")

try:
    test = requests.get("http://localhost:1234/v1/models", timeout=5)
    print("✅ Connected!")
except Exception as e:
    print("❌ Cannot connect to LM Studio")
    print(e)
    exit()

# ==========================
# Load Dataset
# ==========================
print("📂 Loading dataset...")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

posts = None

for item in data:
    if item.get("type") == "table" and item.get("name") == "post":
        posts = item["data"]
        break

if posts is None:
    raise Exception("Post table not found!")

print(f"Loaded {len(posts)} posts")


def classify(index, item):
    """Runs in a worker thread. Returns (index, item_with_label) or raises."""
    raw_text = item.get("txtContent")

    if raw_text is None:
        return index, None

    text = str(raw_text).strip()
    if text == "":
        return index, None

    prompt = f"""
ตอบเพียงตัวเลขเดียว

0 = ปกติ
1 = ไม่ปกติ
2 = ไม่แน่ใจ

ข้อความ:
{text}
"""

    try:
        response = session.post(
            LMSTUDIO_URL,
            json={
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0,
                "max_tokens": 200,
            },
            timeout=300,
        )

        raw = response.json()

        if "choices" not in raw:
            raise ValueError(raw)

        choice = raw["choices"][0]
        msg = choice["message"]
        answer = (msg.get("content") or "").strip()

        if not answer:
            answer = (msg.get("reasoning_content") or "").strip()

        if not answer:
            raise ValueError(f"Empty response from model. Full raw: {json.dumps(raw, ensure_ascii=False)}")

        digit = next((ch for ch in answer if ch in label_map), None)

        if digit is None:
            raise ValueError(f"No valid label digit found in response: {answer!r}")

        label = label_map.get(digit, "ไม่แน่ใจ")
        item["label"] = label
        return index, item

    except Exception as e:
        item["label"] = "ERROR"
        print(f"ERROR on post {item.get('id', '')}: {e}")
        return index, item


# ==========================
# Label Posts (concurrently, results kept in original order)
# ==========================

# Pre-size the results list so we can drop each finished item straight into
# its original slot regardless of which order threads finish in.
results_by_index = [None] * len(posts)
completed = 0
lock = threading.Lock()


def save_partial():
    # Only keep the slots that were actually filled in (skips empty-text posts)
    to_save = [r for r in results_by_index if r is not None]
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(to_save, f, ensure_ascii=False, indent=2)


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(classify, i, item): i
        for i, item in enumerate(posts)
    }

    for future in as_completed(futures):
        i, labeled_item = future.result()

        if labeled_item is not None:
            results_by_index[i] = labeled_item

        with lock:
            completed += 1
            print(f"[{completed}/{len(posts)}]")

            if completed % SAVE_EVERY == 0:
                with save_lock:
                    save_partial()
                print(f"💾 Saved {completed} posts")

# ==========================
# Save Final
# ==========================
with save_lock:
    save_partial()

print("\n🎉 Finished!")
print(f"Saved to {OUTPUT_FILE}")

🔌 Testing LM Studio...
✅ Connected!
📂 Loading dataset...
Loaded 2132 posts
[1/2132]
[2/2132]
[3/2132]
[4/2132]
[5/2132]
[6/2132]
[7/2132]
[8/2132]
[9/2132]
[10/2132]
[11/2132]
[12/2132]
[13/2132]
[14/2132]
[15/2132]
[16/2132]
[17/2132]
[18/2132]
[19/2132]
[20/2132]
[21/2132]
[22/2132]
[23/2132]
[24/2132]
[25/2132]
[26/2132]
[27/2132]
[28/2132]
[29/2132]
[30/2132]
[31/2132]
[32/2132]
[33/2132]
[34/2132]
[35/2132]
[36/2132]
[37/2132]
[38/2132]
[39/2132]
[40/2132]
[41/2132]
[42/2132]
[43/2132]
[44/2132]
[45/2132]
[46/2132]
[47/2132]
[48/2132]
[49/2132]
[50/2132]
[51/2132]
[52/2132]
[53/2132]
[54/2132]
[55/2132]
[56/2132]
[57/2132]
[58/2132]
[59/2132]
[60/2132]
[61/2132]
[62/2132]
[63/2132]
[64/2132]
[65/2132]
[66/2132]
[67/2132]
[68/2132]
[69/2132]
[70/2132]
[71/2132]
[72/2132]
[73/2132]
[74/2132]
[75/2132]
[76/2132]
[77/2132]
[78/2132]
[79/2132]
[80/2132]
[81/2132]
[82/2132]
[83/2132]
[84/2132]
[85/2132]
[86/2132]
[87/2132]
[88/2132]
[89/2132]
[90/2132]
[91/2132]
[92/2132]
[93/2132]
[94/